# LEDGAR — clause retrieval (milestone m2)

> **Summary**
> - Thirteen bank encodings compared on validation (TF-IDF, five off-the-shelf encoders raw and centred, the
>   fine-tuned classifier's features), each with five hybrid variants: 78 candidates. The pre-registered rule kept
>   **one**: TF-IDF cosine, dense. Test macro P@10 **0.658** (validation 0.661), MRR 0.873, 10-NN vote macro-F1 0.737.
> - The within-type gate decided. A degraded clause finds its source with MRR 0.897 under TF-IDF against 0.63-0.76
>   for every dense encoder; the gate at 0.95 x 0.897 removed them all. Dense encoders retrieve the *type* about as
>   well (macro P@10 0.61-0.67, Legal-BERT centred 0.674) but not the *clause*.
> - The classifier's features win the type proxy (P@10 0.758) and lose the clause (MRR 0.508): the neural collapse
>   predicted in the README. Every hybrid variant reaches P@10 0.80-0.81 whatever the encoder, because that is the
>   classifier's accuracy, and loses 8 to 28 points of within-type MRR against its own dense variant: the type
>   should be a filter the lawyer ticks, not one the classifier guesses.
> - For lawyer *questions* the picture reverses, and it is reported, not decisional: contrastive sentence encoders
>   answer type names (P@10 0.53-0.59) and keep the asked constraint in 76-88 % of the retrieved clauses (e5 and
>   MiniLM 86-88 %, lift 4), TF-IDF 0.48 and 75 %, while the raw MLM encoders (Legal-BERT, DistilBERT) score
>   0.02-0.05 on type names and no better than the base rate on constraints.

Given a pasted clause or a lawyer's question, the retriever returns the most similar clauses of a clause bank
(the 60 000 training clauses). The runs were made by `scripts/retrieval.py` (one encoder at a time, validation
only), the configuration was chosen by the pre-registered rule of the README (`--choose`), and the test split was
read once (`--final`). This notebook reads the saved runs and checks a few things live on the data. It is run on
Colab (clone the repository at the head of `main`, `pip install -r requirements.txt && pip install -e .`, then
*Run all*): the live demo encodes the 60 000 bank clauses with MiniLM, a minute on a GPU. Under `FAST_RUN=1` (CI)
it runs on a per-class subset in under a minute.

## Setup

In [ ]:
import json
import re
import warnings
from dataclasses import replace

warnings.filterwarnings("ignore", message="IProgress not found")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub.utils import logging as hub_logging

from ledgar import data, retrieval
from ledgar.config import Config, set_seed
from ledgar.retrieval import Index

hub_logging.set_verbosity_error()
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

cfg = Config.from_env()
set_seed(cfg.seed)
RESULTS, FIGURES = cfg.path("results", "retrieval"), cfg.path(cfg.reports_dir, "figures")
ledgar = data.load_ledgar(cfg)
LABELS = data.label_names(ledgar)
bank_texts, y_bank = ledgar["train"]["text"], np.array(ledgar["train"]["label"])
val_texts, y_val = ledgar["validation"]["text"], np.array(ledgar["validation"]["label"])
METRICS = {p.parent.name: json.loads(p.read_text()) for p in sorted(RESULTS.glob("*/metrics.json"))}
assert METRICS, "no retrieval run under results/retrieval: run `make m2` first"
table = retrieval.retrieval_table(RESULTS).set_index("run")
print(cfg)
print(f"{len(METRICS)} runs: {', '.join(METRICS)}")

## 1. The bank, the queries, and what "similar" means

Bank = the 60 000 training clauses; queries = the 10 000 validation clauses; the test split is read once at the
end. The only label we have is the clause type, so a retrieved clause counts as *relevant* when it has the type
of the query: $P@k$ is the share of the top-$k$ with the right type, MRR the reciprocal rank of the first one,
both averaged over types (macro, as for the classifiers). That proxy measures the type, not the wording inside a
type, which is what a lawyer actually compares. Two checks look inside a type: a degraded copy of a clause must
retrieve its source (section 4), and lawyer questions with a verifiable constraint ("a 30-day notice") are graded
by a regex on the retrieved texts.

In [ ]:
counts = np.bincount(y_bank, minlength=cfg.n_classes)
share = counts / counts.sum()
floor = pd.Series(
    {
        "bank clauses": int(counts.sum()),
        "validation queries": len(y_val),
        "clause types in the bank": int((counts > 0).sum()),
        "random P@10, micro (sum of squared type shares)": float(share @ share),
        "random P@10, macro (mean type share)": float(share[counts > 0].mean()),
        "validation clauses with an exact copy in the bank": float(
            data.seen_in_train(ledgar, "validation").mean()
        ),
    }
)
questions = pd.DataFrame([q.__dict__ for q in retrieval.CONSTRAINED_QUERIES]).drop(columns="example")
display(floor.to_frame("value"))
questions

→ **Consequence:** a retriever that ignores the text scores about 0.01 on macro P@10: every number below is measured
against that floor, not against 0. The questions above are the only "lawyer" queries we can grade without a
lawyer; their constraint is lexical (a notice period, a state, a word), and the regex is said to be crude.

## 2. Which encoder retrieves the right type? (validation)

Six off-the-shelf encoders, each also after centering on the bank mean, and the penultimate features of the
fine-tuned classifier when its run exists (`classifier:*`). The 10-nearest-neighbour vote is also scored as a
classifier (macro-F1), which puts the retriever on the scale of part 1.

In [ ]:
cols = [
    "encoder",
    "centered",
    "p_at_1",
    "p_at_5",
    "p_at_10",
    "mrr_micro",
    "knn_macro_f1",
    "perturb_mrr",
    "type_names_p_at_10",
]
ranking = table[cols].sort_values("p_at_10", ascending=False)
best_run = ranking.index[0]
print(f"best macro P@10 on validation: {best_run} ({ranking.iloc[0]['p_at_10']:.4f})")
ranking.round(4)

In [ ]:
bands = ["band=head", "band=middle", "band=tail"]
rows = {
    run: {b.split("=")[1]: m["val"]["by_group"].get(b, {}).get("p_at_10_macro", np.nan) for b in bands}
    | {"not in train": m["val"]["by_group"].get("seen_in_train=False", {}).get("p_at_10_macro", np.nan)}
    | {"truncated": m["val"]["by_group"].get("truncated=True", {}).get("p_at_10_macro", np.nan)}
    for run, m in METRICS.items()
    if not m["centered"]
}
by_group = pd.DataFrame(rows).T.loc[lambda d: d.index.isin(ranking.index)]
by_group = by_group.loc[ranking.index.intersection(by_group.index)]
fig, ax = plt.subplots(figsize=(9, 4))
by_group[["head", "middle", "tail"]].plot.bar(ax=ax, width=0.8)
ax.set(ylabel="macro P@10 (validation)", title="Macro P@10 by frequency band of the query's type", xlabel="")
ax.legend(title="band", fontsize=8)
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
fig.tight_layout()
if not cfg.fast_run:
    fig.savefig(FIGURES / "retrieval_p10_bands.png", dpi=150)
by_group.round(4)

→ **Consequence:** on the type proxy the encoders sit within seven points of each other: Legal-BERT centred 0.674,
bge centred 0.669, TF-IDF 0.661, e5 0.648, MiniLM 0.630, DistilBERT 0.616; the classifier's own features reach
0.758. Every encoder loses 40 points between the head and the tail of the type distribution and 12 points on
clauses that have no exact copy in the bank. Truncation costs little: 1.8 % of clauses exceed 512 wordpieces, and
MiniLM, which cuts 13.5 % at 256, loses 2.5 points on them. The 10-NN vote is a weaker classifier than the
logistic regression of part 1 (0.743 against 0.827 macro-F1), as expected from a non-parametric vote over 60 000
clauses. None of this separates the encoders; the within-type checks of section 4 do.

## 3. Why: the geometry of the bank embeddings

Five numbers on the 60 000 bank vectors of each encoder. **Anisotropy**: mean cosine between random pairs (0 for
an isotropic cloud, close to 1 in a narrow cone, Ethayarajh 2019). **Effective rank**: $\exp H(\sigma / \sum
\sigma)$ of the singular values (Roy & Vetterli 2007), how many directions carry signal. **Hubness**: skewness of
$N_{10}$, the number of top-10 lists each clause appears in (Radovanović et al. 2010). **Alignment** (mean cosine
of same-type pairs) and **uniformity** ($\log \mathbb{E}\, e^{-2\|u-v\|^2}$, lower is more spread) of Wang &
Isola 2020. Centering subtracts the bank mean: the one-line fix for a cone.

In [ ]:
geo_cols = ["anisotropy", "effective_rank", "hubness_skew", "hubness_max_share", "alignment", "uniformity"]
geometry = pd.DataFrame({run: m["geometry"] for run, m in METRICS.items()}).T[geo_cols]
geometry.insert(0, "p_at_10", table["p_at_10"])
geometry = geometry.loc[ranking.index]
fig, ax = plt.subplots(figsize=(7, 4.5))
dense = geometry.dropna(subset=["effective_rank"])
ax.scatter(
    dense["uniformity"], dense["alignment"], s=40 + 400 * dense["p_at_10"], c=dense["p_at_10"], cmap="viridis"
)
for run, r in dense.iterrows():
    ax.annotate(run, (r["uniformity"], r["alignment"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set(
    xlabel="uniformity (lower = more spread)",
    ylabel="alignment (same-type cosine)",
    title="Bank geometry; size and colour = macro P@10",
)
fig.tight_layout()
if not cfg.fast_run:
    fig.savefig(FIGURES / "retrieval_geometry.png", dpi=150)
geometry.round(3)

In [ ]:
full = data.load_ledgar(
    replace(cfg, n_train_per_class=None, n_eval_per_class=None)
)  # the runs saw the whole bank
keys = pd.Series([data.normalise(t) for t in full["train"]["text"]])
choice_path = RESULTS / "choice.json"
shown = json.loads(choice_path.read_text())["run"] if choice_path.exists() else "tfidf"
hubs = pd.read_csv(RESULTS / shown / "hubs.csv")
hubs["type"] = [LABELS[k] for k in hubs["label"]]
hubs["copies in bank"] = [int((keys == keys.iloc[i]).sum()) for i in hubs["index"]]
print(f"the 10 largest hubs of the chosen run, {shown}")
hubs[["index", "type", "n_k", "copies in bank", "text"]]

→ **Consequence:** the raw encoders live in a cone: mean cosine between *random* clauses 0.88 (DistilBERT), 0.82
(e5), 0.80 (Legal-BERT), 0.62 (bge), against 0.33 for MiniLM and 0.03 for TF-IDF. Their raw "alignment" of 0.87-0.92
is that cone, not type structure: after centring it falls to 0.28-0.39 and then tracks P@10 (Legal-BERT 0.39 first,
e5 0.28 last among the dense). Centring is worth about one point for bge, Legal-BERT, MiniLM and DistilBERT and
costs e5 1.6 points: e5 was trained with query/passage prefixes, and the mean it loses carries that asymmetry.
Hubness is the TF-IDF weakness: skewness 4.7 against 1.4-2.1 for the dense encoders, and the top hub, the boilerplate
"This Agreement constitutes the entire agreement between the parties...", sits in 414 of the 60 000 top-10 lists;
bge, DistilBERT and Legal-BERT share that boilerplate hub three to four times less often, MiniLM's and e5's
largest hubs are other types (Indemnifications, Terms, Base Salary).

## 4. Inside a type: a degraded clause must find its source, and lawyer questions must keep their constraint

**Perturbation.** 1 000 validation clauses with no copy in the bank, each cut to its first half, its second half,
or with one sentence removed. The degraded clause is the query; its source is scored like a bank clause; the rank
of the source among bank + source gives Hit@1 and MRR. A retriever that only knows the type cannot do this.
**Questions.** For each of the 18 questions: share of the top-10 with the target type, share of those that
satisfy the constraint, and the base rate of the constraint among the bank clauses of that type (lift =
precision / base rate).

In [ ]:
pert = pd.DataFrame(
    {
        run: {"MRR": m["val"]["perturbation"]["mrr"], "Hit@1": m["val"]["perturbation"]["hit_at_1"]}
        | {f"MRR {v}": s["mrr"] for v, s in m["val"]["perturbation"]["views"].items()}
        for run, m in METRICS.items()
    }
).T.loc[ranking.index]
pert.round(4)

In [ ]:
for run in ("e5", "tfidf"):
    summary = METRICS[run]["val"]["constrained_summary"]
    print(f"{run}: " + ", ".join(f"{k} {v:.3f}" for k, v in summary.items() if k != "n"))
cq = pd.DataFrame(METRICS["e5"]["val"]["constrained"]).set_index("question")
cq_tfidf = pd.DataFrame(METRICS["tfidf"]["val"]["constrained"]).set_index("question")
cols = ["type_p_at_k", "constraint_precision", "lift"]
side = pd.concat({"e5": cq[cols], "tfidf": cq_tfidf[cols]}, axis=1)
side.insert(0, "base_rate", cq["base_rate"])
side.round(2)

In [ ]:
# Live: the 30-day question, with and without the type the lawyer would tick (no classifier involved)
spec = retrieval.ENCODERS["minilm"]
encoder = retrieval.load_encoder(spec, cfg, bank_texts)
E_bank = retrieval.cached_embeddings(encoder, bank_texts, "train", "doc", cfg)
index = Index(E_bank, y_bank, np.arange(len(y_bank)), device=cfg.device)
question = retrieval.CONSTRAINED_QUERIES[0]
pattern = re.compile(question.pattern, flags=re.IGNORECASE)
q = encoder.encode([question.question], "nl")
manual = np.zeros((1, cfg.n_classes), dtype=np.float32)
manual[0, LABELS.index(question.target)] = 1.0


def show(positions, scores):
    return pd.DataFrame(
        {
            "score": scores[0].round(3),
            "type": [LABELS[y_bank[p]] for p in positions[0]],
            "30 days": [bool(pattern.search(bank_texts[p])) for p in positions[0]],
            "clause": [bank_texts[p][:110] + "…" for p in positions[0]],
        }
    )


scores_d, pos_d = index.search(q, 5)
scores_h, pos_h = index.search(q, 5, type_probs=manual, hard=True)
print(f"'{question.question}' — dense top-5, then restricted to {question.target}")
display(show(pos_d, scores_d))
show(pos_h, scores_h)

→ **Consequence:** this is where the encoders separate. A clause cut in half finds itself under TF-IDF with MRR 0.86
(first half) and 0.92 (second half), 0.94 with a sentence removed; MiniLM, e5 and bge manage 0.76-0.81 on the
first half, DistilBERT and Legal-BERT 0.58-0.64, every dense encoder 0.60-0.70 on the second half and 0.85-0.91 with
a sentence removed. The second half is the hard case: the opening
words carry the type ("This Agreement shall be governed by..."), the rest is the specific wording, which a lexical
index keeps and a sentence embedding blurs. The classifier's features collapse there (second half 0.26): they know
the type, not the clause. On the lawyer questions the order reverses: e5 and MiniLM keep the asked constraint in
86-88 % of the retrieved clauses of the right type (lift 4 over the base rate; bge 76-81 %), TF-IDF 75 % (lift
3.3), Legal-BERT 49 %, no better than the base rate. The means are taken over the questions where at least one
clause of the target type was retrieved, so their denominators differ a little between encoders. For "a 30-day notice period" e5 returns 6 Terminations out of 10, all with 30 days
(base rate 4 %, lift 23); TF-IDF 3 out of 10. Three questions fail for everyone: "indemnification covering
attorneys' fees" lands on Expenses and Fees clauses, "confidentiality surviving two years" has almost no match in
the bank (base rate 0.5 %), "interest on late payments" lands on Interests. The questions are not a benchmark, 18
items of which 16 carry a pattern, graded by regex, but they show what the proxy cannot: the type is not the
wording.

## 5. The hybrid, and the pre-registered choice

The classifier of part 1 predicts the type of the query; the bank is then restricted to that type (*hard*) or
re-scored with $\cos + \lambda \log p(\text{type})$ (*soft*). On the type proxy the hard filter cannot lose: its
P@10 is the classifier's accuracy on the type. The perturbation MRR is what it costs inside a type: a wrong
predicted type removes the source itself. The rule written before the runs: among every (encoder, centering,
variant), keep those whose perturbation MRR is at least 95 % of the best one, then take the best macro P@10;
ties go to the simpler variant.

In [ ]:
rows = []
for run, m in METRICS.items():
    v = m["val"]
    rows.append(
        {"run": run, "variant": "dense", "p10": v["clause"]["p_at_10_macro"], "mrr": v["perturbation"]["mrr"]}
    )
    for variant, h in v["hybrid"].items():
        rows.append(
            {
                "run": run,
                "variant": variant,
                "p10": h["clause"]["p_at_10_macro"],
                "mrr": h["perturbation"]["mrr"],
            }
        )
grid = pd.DataFrame(rows)
threshold = 0.95 * grid["mrr"].max()
fig, ax = plt.subplots(figsize=(8, 5))
markers = {"dense": "o", "hard": "x"}
for run, g in grid.groupby("run"):
    sc = ax.scatter(g["mrr"], g["p10"], s=30, label=run, marker="o", alpha=0.8)
    for _, r in g[g["variant"] == "hard"].iterrows():
        ax.scatter(r["mrr"], r["p10"], s=60, marker="x", color=sc.get_facecolor()[0])
ax.axvline(threshold, color="#b91c1c", ls="--", lw=1)
ax.text(threshold, ax.get_ylim()[0], " gate: 0.95 × best MRR", color="#b91c1c", fontsize=8, va="bottom")
ax.set(
    xlabel="perturbation MRR (validation)",
    ylabel="macro P@10 (validation)",
    title="Every candidate: dense (o), soft λ (o), hard (x)",
)
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
if not cfg.fast_run:
    fig.savefig(FIGURES / "retrieval_hybrid.png", dpi=150)
grid.pivot(index="run", columns="variant", values=["p10", "mrr"]).loc[ranking.index].round(4)

In [ ]:
choice_path = RESULTS / "choice.json"
if choice_path.exists():
    choice = json.loads(choice_path.read_text())
    n = f"{choice['n_gated']} of {choice['n_candidates']}"
    print(
        f"chosen on {choice['date_utc'][:10]}: {choice['run']} / {choice['variant']} "
        f"(P@10 {choice['p10']:.4f}, perturbation MRR {choice['mrr']:.4f}; "
        f"{n} candidates kept by the gate {choice['threshold']:.4f})"
    )
else:
    provisional = retrieval.choose(
        grid.assign(centered=[METRICS[r]["centered"] for r in grid["run"]]).to_dict("records")
    )
    print("no choice.json yet (the rule waits for the classifier:* run); on the runs so far it would pick:")
    print({k: provisional[k] for k in ("run", "variant", "p10", "mrr", "n_gated", "n_candidates")})

→ **Consequence:** every hybrid variant of every encoder lands between 0.80 and 0.81 of macro P@10, with a hard
filter or a soft prior as weak as λ = 0.1: that number is the LoRA classifier's type accuracy, not retrieval. The
price is paid inside the type: perturbation MRR falls from 0.90 to 0.78 (TF-IDF, soft 0.1) and from 0.75 to 0.53
(bge, hard), 8 to 28 points against each encoder's own dense variant, because a wrong predicted type (one
validation query in eight, one in five per type) removes the source clause itself. The rule,
written before the runs, kept 1 of the 78 candidates: TF-IDF dense (gate 0.852; the best hybrid, TF-IDF soft 0.1,
is at 0.778). Had TF-IDF not been a candidate, the gate would have been 0.723 and bge centred the choice: a
sensitivity note, not a decision. The type information is worth having, but as a filter the lawyer ticks (the live
demo of section 4) rather than one the classifier guesses.

## 6. The test split, once; limits; what the retriever must not be used for

The chosen variant and the dense variant of the same run are the two final models. The test split (10 000
clauses) was read once by `scripts/retrieval.py --final`; the paired bootstrap resamples the same queries for
both.

In [ ]:
finals = {run: m["test"] for run, m in METRICS.items() if "test" in m}
if finals:
    run, t = next(iter(finals.items()))
    variants = [k for k, v in t.items() if isinstance(v, dict) and "clause" in v]
    test_table = pd.DataFrame({v: t[v]["clause"] for v in variants}).T
    print(f"{run}: chosen variant {t['chosen_variant']}")
    if "comparison" in t:
        c = t["comparison"]
        verdict = "significant" if c["significant"] else "not significant"
        print(
            f"macro P@10 difference {t['chosen_variant']} - dense: {c['diff']:+.4f}, "
            f"95 % CI [{c['low']:+.4f}, {c['high']:+.4f}] -> {verdict} at 5 %"
        )
    groups = pd.DataFrame({v: {g: s["p_at_10_macro"] for g, s in t[v]["by_group"].items()} for v in variants})
    display(test_table[["p_at_1_macro", "p_at_5_macro", "p_at_10_macro", "mrr", "knn_macro_f1"]].round(4))
    display(groups.round(4))
else:
    print("the test split has not been read yet (run --choose, then --final)")

→ **Consequence:** the chosen configuration holds on the test split: macro P@10 0.658 on the 10 000 test clauses
against 0.661 on validation, MRR 0.873, 10-NN vote macro-F1 0.737 (accuracy 0.826). The same gradient as on validation: head
types 0.83, middle 0.70, tail 0.43; clauses with an exact copy in the bank 0.78, the others 0.66. The chosen variant
is dense, so there is no second final model and no paired test. What the retrieval part leaves open, and would need
its own pre-registration: a dense encoder (e5) as the entry point for *questions*, TF-IDF for *pasted clauses*, and
the type as a manual filter.

**Limits, and what the retriever must not be used for.**
- It retrieves clauses whose *wording* is close, under a proxy that only checks the clause type and two
  within-type tests that no lawyer validated. **It does not establish legal equivalence**: a retrieved clause is
  not a precedent, and a missing clause is not evidence that none exists.
- The bank is LEDGAR: clauses of SEC filings, in English, US-style drafting. Other jurisdictions, languages and
  contract families are out of distribution; the bias audit (part 4) measures how far.
- Clauses longer than the encoder's window (256 or 512 wordpieces) are read truncated; the per-group table of
  section 2 shows what that costs.
- The hybrid inherits every error of the classifier: when it filters hard on a wrong type, the right clauses
  are unreachable. The soft variant is the one the rule allows.
- The constraint grading is a regex: "30 days" matched for the wrong reason counts as a hit.

## Files used

`results/retrieval/<encoder>[+center]/` (configuration, metrics, manifest, `topk_val.npz`, `pred_val.csv`,
`hubs.csv`), `results/retrieval/choice.json`, the chosen run's `topk_test.npz` and `pred_test.csv`;
`reports/retrieval_table.md`. Figures: `reports/figures/retrieval_p10_bands.png`, `retrieval_geometry.png`,
`retrieval_hybrid.png`.